# Event Match Report


In [ ]:
# Parameters are injected by smk/scripts/_4_event_match_notebook.py.
EVENT = None
DIAGNOSTICS_FP_L = []
STATUS_FP_L = []
OUTPUT_DIR = None
CSDA_DATES_FP = None


In [ ]:
from pathlib import Path
import html
import json

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

plt.rcParams['figure.facecolor'] = 'white'


In [ ]:
def _read_raster(fp):
    """Read a raster into a float array and minimal geospatial metadata."""
    with rasterio.open(fp) as ds:
        arr = ds.read(masked=True).astype('float32').filled(np.nan)
        meta_d = {
            'height': ds.height,
            'width': ds.width,
            'count': ds.count,
            'transform': ds.transform,
            'crs': ds.crs,
        }
    return arr, meta_d


def _normalize_band(band_arr, low=2, high=98):
    """Normalize one band for plotting."""
    valid_bx = np.isfinite(band_arr)
    out_arr = np.full(band_arr.shape, np.nan, dtype='float32')
    if valid_bx.sum() == 0:
        return out_arr
    low_v = np.nanpercentile(band_arr[valid_bx], low)
    high_v = np.nanpercentile(band_arr[valid_bx], high)
    if (not np.isfinite(low_v)) or (not np.isfinite(high_v)) or high_v <= low_v:
        out_arr[valid_bx] = 0.0
        return out_arr
    band_arr = np.clip(band_arr, low_v, high_v)
    out_arr[valid_bx] = (band_arr[valid_bx] - low_v) / (high_v - low_v)
    return out_arr


def _rgb_plot_arr(arr):
    """Build a visible RGB array from a PlanetScope-like raster."""
    if arr.shape[0] >= 3:
        rgb_arr = np.stack([_normalize_band(arr[2]), _normalize_band(arr[1]), _normalize_band(arr[0])], axis=-1)
    elif arr.shape[0] == 1:
        gray_arr = _normalize_band(arr[0])
        rgb_arr = np.stack([gray_arr, gray_arr, gray_arr], axis=-1)
    else:
        rgb_arr = np.stack([_normalize_band(arr[i]) for i in range(min(3, arr.shape[0]))], axis=-1)
    return np.nan_to_num(rgb_arr, nan=0.0)


def _label_plot_arr(label_arr):
    """Build an RGB plot array for FloodPlanet label tiles."""
    class_color_d = {
        0: np.array([0.82, 0.82, 0.82], dtype='float32'),
        1: np.array([0.94, 0.94, 0.90], dtype='float32'),
        2: np.array([0.12, 0.47, 0.71], dtype='float32'),
    }
    plot_arr = np.zeros((*label_arr.shape, 3), dtype='float32')
    for value, color_arr in class_color_d.items():
        plot_arr[label_arr == value] = color_arr
    plot_arr[~np.isin(label_arr, list(class_color_d))] = np.array([0.55, 0.55, 0.55], dtype='float32')
    return plot_arr


def _warp_to_reference_grid(src_arr, src_meta_d, ref_meta_d):
    """Warp a candidate raster onto the reference raster grid for aligned plotting."""
    dst_arr = np.full((src_arr.shape[0], ref_meta_d['height'], ref_meta_d['width']), np.nan, dtype='float32')
    for band_i in range(src_arr.shape[0]):
        reproject(
            source=src_arr[band_i],
            destination=dst_arr[band_i],
            src_transform=src_meta_d['transform'],
            src_crs=src_meta_d['crs'],
            src_nodata=np.nan,
            dst_transform=ref_meta_d['transform'],
            dst_crs=ref_meta_d['crs'],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return dst_arr


def _fmt_ts(value):
    """Format a timestamp-like value for plot titles."""
    if value is None or pd.isna(value):
        return 'NA'
    return pd.Timestamp(value).strftime('%Y-%m-%d %H:%M')


def _timestamp_ns(value):
    """Convert a timestamp-like value to UTC nanoseconds for relative positioning."""
    if value is None or pd.isna(value):
        return None
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        ts = ts.tz_localize('UTC')
    else:
        ts = ts.tz_convert('UTC')
    return ts.value


def _attempt_sort_key(attempt_d):
    """Sort attempts by scene datetime then item id."""
    acquired = pd.Timestamp(attempt_d.get('acquired')) if attempt_d.get('acquired') else pd.Timestamp.max.tz_localize('UTC')
    return acquired, str(attempt_d.get('item_id'))


class _NotebookHeading:
    """Small HTML object used for notebook display headings."""
    def __init__(self, text):
        self.text = text

    def _repr_html_(self):
        return f"<h2>{html.escape(self.text)}</h2>"

    def __str__(self):
        return self.text


def _chip_heading(match_d):
    """Build the notebook subheading text for one chip plot."""
    chip_context = match_d['chip_context']
    csda_date = _fmt_ts(CSDA_DATE_D.get(chip_context['event'], chip_context.get('reference_datetime')))
    return _NotebookHeading(f"{chip_context['event']} | {chip_context['chip_id']} | CSDA {csda_date}")


def _chip_summary_df(match_d):
    """Build a compact candidate summary table for one chip."""
    row_l = []
    chip_id = match_d['chip_context']['chip_id']
    attempt_l = [d for d in sorted(match_d.get('attempts', []), key=_attempt_sort_key) if d.get('raster_fp')]
    for attempt_d in attempt_l:
        metrics_d = attempt_d.get('compare_metrics') or {}
        row_l.append({
            'chip_id': chip_id,
            'item_id': attempt_d.get('item_id'),
            'match_score': metrics_d.get('match_mean_quad_corr'),
            'rank': attempt_d.get('match_candidate_rank'),
            'matched': bool(metrics_d.get('matched', False)),
        })
    if not row_l:
        row_l.append({'chip_id': chip_id, 'item_id': None, 'match_score': None, 'rank': None, 'matched': False})
    return pd.DataFrame(row_l)


def _add_csda_line(fig, ax_ar, attempt_l, fetch_col_offset, label_col, csda_value):
    """Add a figure-level line showing the CSDA datetime position relative to fetched chips."""
    target_ns = _timestamp_ns(csda_value)
    time_l = [(idx + fetch_col_offset, _timestamp_ns(d.get('acquired'))) for idx, d in enumerate(attempt_l)]
    time_l = [(col, ts) for col, ts in time_l if ts is not None]
    if target_ns is None or not time_l:
        line_col = fetch_col_offset if attempt_l else 0
    elif target_ns <= time_l[0][1]:
        line_col = time_l[0][0]
    elif target_ns >= time_l[-1][1]:
        line_col = time_l[-1][0]
    else:
        line_col = time_l[-1][0]
        for (left_col, left_ns), (right_col, right_ns) in zip(time_l[:-1], time_l[1:]):
            if left_ns <= target_ns <= right_ns:
                fraction = (target_ns - left_ns) / max(right_ns - left_ns, 1)
                line_col = left_col + fraction * (right_col - left_col)
                break
    fig.canvas.draw()
    left = ax_ar[0, 0].get_position().x0
    right = ax_ar[0, label_col].get_position().x1
    bottom = ax_ar[1, 0].get_position().y0
    top = ax_ar[0, 0].get_position().y1
    x = left + ((line_col + 0.5) / ax_ar.shape[1]) * (right - left)
    fig.add_artist(Line2D([x, x], [bottom, top], transform=fig.transFigure, color='black', linewidth=2.5))


def _plot_chip(match_d):
    """Plot one chip with fetched rasters, label tile, and explicit unmatched placement."""
    chip_context = match_d['chip_context']
    attempt_l = [d for d in sorted(match_d.get('attempts', []), key=_attempt_sort_key) if d.get('raster_fp')]
    matched_idx = None
    for idx, attempt_d in enumerate(attempt_l):
        if (attempt_d.get('compare_metrics') or {}).get('matched', False):
            matched_idx = idx
            break
    unmatched = matched_idx is None
    fetch_col_offset = 1 if unmatched else 0
    ncols = max(len(attempt_l) + fetch_col_offset + 1, 2)
    ref_col = 0 if unmatched else matched_idx
    label_col = ncols - 1
    reference_fp = Path(match_d['reference_fp'])
    label_fp = reference_fp.parents[1] / 'labels' / reference_fp.name
    ref_arr, ref_meta_d = _read_raster(reference_fp)
    fig, ax_ar = plt.subplots(2, ncols, figsize=(max(3.6 * ncols, 7.0), 6.8), squeeze=False, constrained_layout=True)
    csda_date = _fmt_ts(CSDA_DATE_D.get(chip_context['event'], chip_context.get('reference_datetime')))
    for ax in ax_ar.ravel():
        ax.set_axis_off()
    ax_ar[0, ref_col].imshow(_rgb_plot_arr(ref_arr), interpolation='nearest', aspect='equal')
    if unmatched:
        ax_ar[0, ref_col].add_patch(Rectangle((-.5, -.5), ref_meta_d['width'], ref_meta_d['height'], fill=False, edgecolor='#d62728', linewidth=4.0))
        ax_ar[0, ref_col].text(0.5, -0.08, 'UNMATCHED', color='#d62728', ha='center', va='top', transform=ax_ar[0, ref_col].transAxes, fontsize=10, fontweight='bold')
    if not attempt_l:
        ax_ar[1, 0].text(0.5, 0.5, 'no fetched rasters', ha='center', va='center', transform=ax_ar[1, 0].transAxes)
    for idx, attempt_d in enumerate(attempt_l):
        plot_col = idx + fetch_col_offset
        fetch_fp = Path(attempt_d['raster_fp'])
        if not fetch_fp.exists():
            ax_ar[1, plot_col].text(0.5, 0.5, 'missing raster', ha='center', va='center', transform=ax_ar[1, plot_col].transAxes)
            continue
        fetch_arr, fetch_meta_d = _read_raster(fetch_fp)
        fetch_arr = _warp_to_reference_grid(fetch_arr, fetch_meta_d, ref_meta_d)
        metrics_d = attempt_d.get('compare_metrics') or {}
        matched = bool(metrics_d.get('matched', False))
        ax_ar[1, plot_col].imshow(_rgb_plot_arr(fetch_arr), interpolation='nearest', aspect='equal')
        for spine in ax_ar[1, plot_col].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2.0 if matched else 0.8)
            spine.set_edgecolor('#2ca02c' if matched else '#777777')
    if label_fp.exists():
        label_arr, _ = _read_raster(label_fp)
        label_band_arr = label_arr[0] if label_arr.ndim == 3 else label_arr
        ax_ar[1, label_col].imshow(_label_plot_arr(label_band_arr), interpolation='nearest', aspect='equal')
    else:
        ax_ar[1, label_col].text(0.5, 0.5, 'label missing', ha='center', va='center', transform=ax_ar[1, label_col].transAxes)
    _add_csda_line(fig, ax_ar, attempt_l, fetch_col_offset, label_col, CSDA_DATE_D.get(chip_context['event'], chip_context.get('reference_datetime')))
    return fig


In [ ]:
CSDA_DATE_D = json.loads(Path(CSDA_DATES_FP).read_text())
match_d_l = [json.loads(Path(fp).read_text()) for fp in DIAGNOSTICS_FP_L]
status_d_l = [json.loads(Path(fp).read_text()) for fp in STATUS_FP_L]
status_df = pd.DataFrame(status_d_l)
display(status_df[['event', 'chip_id', 'status', 'matched', 'attempted_candidates']].sort_values(['event', 'chip_id']))


In [ ]:
for match_d in sorted(match_d_l, key=lambda d: d['chip_context']['chip_id']):
    display(_chip_heading(match_d))
    display(_chip_summary_df(match_d))
    display(_plot_chip(match_d))
